A trace represents the complete journey of one request through your system.

conceptually:

``` markdown
TRACE
│
├── Supervisor
│
├── SQL Agent
│   ├── LLM
│   └── SQL Tool
│
├── Analyst
│   └── LLM
│
└── Response
```

So :
``` markdown
Trace = Complete excecution journey of a request.
```

#### Trace ID

Every trace needs an identifier. that identifier is called Trace id 

example : trace_id = "8f72a1c9-42b3-4c91"

suppose we have 3 user send requests:
``` markdown
User A → Trace ID: 111aaa

User B → Trace ID: 222bbb

User C → Trace ID: 333ccc
```
now logs can be connected 

for user A:

``` markdown
111aaa
 ├── Supervisor
 ├── SQL Agent
 ├── LLM
 ├── SQL Tool
 └── Response
```

for user B

``` markdown
222bbb
 ├── Supervisor
 ├── Analyst
 └── Response
```

The Tracer ID Answers: Which execution does this event belong to.

#### Span

A trace tells you the whole journey.

A span tells you about one operation within that journey.

for example:
``` markdown
Trace
│
├── Supervisor       ← Span
├── SQL Agent        ← Span
├── LLM Call         ← Span
├── SQL Tool         ← Span
└── Analyst          ← Span
```
Here 

Trace = entire request

Span = Individual operation inside the request.

#### Span ID

It is similar to the trace id
each trace it has trace id and each span has its span id

for example 

for Trace id = T123

Spans :
``` markdown
Supervisor
Span ID = S001

SQL Agent
Span ID = S002

LLM
Span ID = S003

SQL Tool
Span ID = S004
```

Trace ID vs Span ID

| Concept        | Meaning                                                     |
| -------------- | ----------------------------------------------------------- |
| Trace ID       | Identifies the entire request/execution                     |
| Span ID        | Identifies one operation                                    |
| Parent Span ID | Identifies the operation that created the current operation |


The Span Tree

A trace is commonly visualized as tree.

``` markdown
Trace T123
│
└── Supervisor
    │
    ├── SQL Agent
    │   │
    │   ├── LLM
    │   │
    │   └── SQL Tool
    │       │
    │       └── Database Query
    │
    ├── Analyst Agent
    │   │
    │   └── LLM
    │
    └── Visualization Agent
        │
        └── LLM
```

#### Span Timings 

Every span usually contains timing information.

for example: 
``` markdown
SQL Agent

Start: 10:00:01.000
End:   10:00:04.500

Duration = 3.5 seconds
```

the trace could be like:

``` markdown 
Supervisor       5.0 sec
│
├── SQL Agent    3.5 sec
│   ├── LLM      1.8 sec
│   └── SQL      1.2 sec
│
├── Analyst      1.0 sec
│   └── LLM      0.8 sec
│
└── Visualization 0.9 sec
    └── LLM       0.7 sec
```

#### Latency

Latency = how long an operation takes.

for example:

``` markdown
LLM call latency = 2.3 sec
Database latency = 1.2 sec
Agent latency = 4.0 sec
Total request latency = 5.4 sec
```

#### Run

especially in LLM/agent observability systems.
A run generally represents an execution of some component

example 
``` markdown
User Request
    ↓
Supervisor Run
    ↓
SQL Agent Run
    ↓
LLM Run
    ↓
Tool Run
```

A run can represent:
* LLM invocation
* Chain Execution
* Agent Execution
* Tool Excecution
* Retrieval Execution
* workflow execution


A run represents an execution of a component; tracing systems commonly represent such executions as spans or span-like records.

#### Session

Conversation:

``` markdown
User:
What were sales last month?

AI:
₹10 crore

User:
What about the previous month?

AI:
₹8 crore

User:
Why did sales increase?

AI:
...
```

The above Complete conversation is called a session. for each conversation we need to give a session id an group them. Session ID = SESSION_123


``` markdown
Session
│
├── Trace 001
│   └── "What were sales last month?"
│
├── Trace 002
│   └── "What about previous month?"
│
└── Trace 003
    └── "Why did sales increase?"
```

The Heirarchy

``` markdown
Session
   │
   ├── Trace
   │    ├── Span
   │    ├── Span
   │    └── Span
   │
   ├── Trace
   │    ├── Span
   │    └── Span
   │
   └── Trace
        └── Span
```

#### Metadata

Additional Information.

``` json
{
  "model": "gpt-5",
  "temperature": 0.2,
  "environment": "production",
  "user_id": "user_123",
  "request_id": "req_789"
}
```

Metadata Provides context about the Execution.

#### Tags

Tags are labels attached to traces/runs/spans to make filtering and analysis easier.

for example: 

``` markdown
environment=production
team=finance
agent=sql
model=gpt-5
region=india
```

Tags are particularly useful for dashboards and filtering.


#### Metadata vs Tags

Metadata : Detaild Contextual information.

``` markdown 
model = gpt-5
temperature = 0.2
prompt_version = v17
customer_plan = enterprise
```

Tags : which are useful for categorical labels.

``` markdown
production
sql-agent
finance
high-priority
```

The exact Distinction depends on the observability platform, but conceptually:

``` markdown
Metadata → information
Tags → classification/filtering
```



#### Correlation ID

A correlation ID is used to connect related events across different components/services.

Suppose:

``` markdown
API
 ↓
Agent Service
 ↓
LLM Service
 ↓
Database Service
```

The same correlation ID can travel through the system:


correlation_id = CORR-123

Then:

``` markdown
API:
CORR-123

Agent:
CORR-123

LLM:
CORR-123

Database:
CORR-123
```

Now we can correlate logs across services.



#### Request ID

A request ID identifies a particular request.

example : request_id = req-98231

when api recives: Post/Chat

it genrates request_id = req-98231

Then logs might contain:
``` markdown
[req-98231] Request received
[req-98231] Supervisor started
[req-98231] SQL Agent started
[req-98231] SQL executed
[req-98231] Response generated
```

#### User ID
``` markdown
user_id = user_456
```

This identifies the user who initiated the request.

example 
``` markdown
Trace
│
├── user_id = user_456
├── session_id = session_100
├── request_id = req_900
└── trace_id = trace_123
```

This allows questions like:

Which user experienced the failure?, How many requests did this user make?, Which users are consuming the most tokens?

**Important**: User IDs should be handled carefully because observability data can contain sensitive information. Avoid putting unnecessary personal data into traces.

##### Environment

Every application should distinguish environments.

Typical environments:
``` markdown
development
staging
production
```

Example : environment = production

Why this is important?

imagine:

``` markdown
Trace 1 → development
Trace 2 → staging
Trace 3 → production
```

If you don't track environment, you might accidentally analyze development traffic as production traffic.

#### Model Information

For LLM Applications, model information is Extremely important.

A model span might contain: 

``` markdown
provider = OpenAI
model = GPT-5
temperature = 0.2
max_tokens = 2000
```
provider = Anthropic
model = Claude

Different models can have different:
* Latency
* cost
* quality
* token limits
* failure rates

you may discover:
``` markdown
GPT-5
Average latency = 1.8 sec

Model B
Average latency = 4.2 sec
```
``` markdown
Model A
Cost/request = ₹0.40

Model B
Cost/request = ₹1.20
```

Now tracing becomes useful for cost and performance optimization.



#### Token Information

LLM calls should ideally capture token usage.

``` markdown
Input tokens:
1200

Output tokens:
350

Total:
1550
```

A trace could show:

``` markdown
LLM Call

Model: GPT-5

Input tokens: 1200
Output tokens: 350
Total tokens: 1550

Latency: 1.8 sec
```

Now you can analyze:
``` markdown
Which requests consume the most tokens?
or 
Which agent has the highest token usage?
or
Why did this request become expensive?
```

#### Token Usage Across a Trace

``` markdown
Supervisor
│
├── SQL Agent
│   └── LLM
│       Input: 1000
│       Output: 300
│
├── Analyst Agent
│   └── LLM
│       Input: 1500
│       Output: 500
│
└── Visualization Agent
    └── LLM
        Input: 800
        Output: 200
```

Total:

``` markdown
Input:
1000 + 1500 + 800
= 3300

Output:
300 + 500 + 200
= 1000

Total:
4300 tokens
```
This allows you to understand the token economics of a workflow.


#### Putting Everything Together

``` markdown
if user asks: "Show me sales performance for the last quarter."

The application runs:

Trace ID: T12345
Session ID: S456
Request ID: R789
User ID: U001
Environment: production

Then:
Trace T12345
│
└── Supervisor
    │
    ├── SQL Agent
    │   │
    │   ├── LLM
    │   │   ├── Model: GPT-5
    │   │   ├── Input Tokens: 900
    │   │   ├── Output Tokens: 250
    │   │   └── Latency: 1.2 sec
    │   │
    │   └── SQL Tool
    │       ├── Query Execution
    │       └── Latency: 2.0 sec
    │
    ├── Analyst Agent
    │   │
    │   └── LLM
    │       ├── Model: GPT-5
    │       ├── Input Tokens: 1200
    │       ├── Output Tokens: 400
    │       └── Latency: 1.5 sec
    │
    └── Visualization Agent
        │
        └── LLM
            ├── Model: GPT-5
            ├── Input Tokens: 700
            ├── Output Tokens: 200
            └── Latency: 0.9 sec
```
